In [2]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

raw = pd.read_excel("../Data/10YT.xlsx")
raw = raw[~raw.iloc[:, 1].astype(str).str.strip().str.lower().eq("close")]

ric_to_country = {"GB10YT=RR":"United Kingdom","US10YT=RR":"United States","DE10YT=RR":"Germany",
    "FR10YT=RR":"France","IT10YT=RR":"Italy","JP10YT=RR":"Japan","CA10YT=RR":"Canada"}
raw = raw.rename(columns={raw.columns[0]:"date"})
raw.columns = ["date"] + [c.split(" ")[0] for c in raw.columns[1:]]
raw = raw.rename(columns=ric_to_country)

raw["date"] = pd.to_datetime(raw["date"], dayfirst=True, errors="coerce")
long = raw.melt(id_vars="date", var_name="country", value_name="yield")
long["yield"] = pd.to_numeric(long["yield"], errors="coerce")
long = long.dropna(subset=["date","yield"]).sort_values("date")
long = (long.set_index("date").groupby("country")["yield"].resample("ME").last()
        .reset_index().dropna(subset=["yield"]))
long = long[long.country.isin(ric_to_country.values())]

others = long[long.country != "United Kingdom"]
uk = long[long.country == "United Kingdom"]

# grey context lines (all other G7), one per country
context = alt.Chart(others).mark_line(strokeWidth=1, color="#C9C9C9").encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("yield:Q", title="10-year sovereign bond yield",
            axis=alt.Axis(labelExpr="datum.label + '%'")),
    detail="country:N")

# UK: bold, house red
uk_line = alt.Chart(uk).mark_line(strokeWidth=2.4, color="#E6224B").encode(
    x="date:T", y="yield:Q")

uk_label = alt.Chart(uk).mark_text(
    align="left", dx=6, fontSize=13, fontWeight="bold", color="#E6224B").encode(
    x=alt.X("date:T", aggregate="max"),
    y=alt.Y("yield:Q", aggregate={"argmax": "date"}),
    text=alt.value("UK"))

caption = alt.Title(
    text="Source: LSEG",
    subtitle=["UK 10-year yield against other G7 economies, %. Monthly, 1990–2026."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (context + uk_line + uk_label)
    .properties(width=560, height=340, padding={"right":28}, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, name="g7_yields_uk", svg=True)
chart

alt.LayerChart(...)